# WaferGuard — Causality-Safe Supervised Autoencoder

This notebook trains and evaluates a supervised autoencoder on `secom_telemetry_modified.csv`. Place the authorised dataset beside the notebook and run all cells.

**Data provenance:** the bundled CSV is a clearly documented fallback derived from the public UCI SECOM dataset because the task-specific organiser file was not retrievable. A deterministic 20% subset was assigned the PS-L sentinel and firmware-code semantics specified in the brief. It is suitable for exercising the code path, but it is not represented as the organiser-provided modified dataset. Replace it with the exact assessment CSV if the organisers provide it.

## Checks built into the pipeline

- Contiguous chronological train, validation, and test windows.
- Imputation, scaling, and numeric-feature selection based on the training window only.
- Weighted class sampling in training; validation and test rows remain unchanged.
- Explicit per-wafer loss shapes to prevent unintended tensor broadcasting.
- PS-L sentinel and firmware-code handling that preserves the station invariant.
- Decision threshold selected on validation data and frozen before test evaluation.
- Early stopping based on validation classification loss, with bounded standardized values to reduce sensitivity to extreme measurements.

**Ordering assumption:** if no parseable timestamp or sequence is available, CSV row order is used as acquisition order. Confirm this against the dataset source before production use.


In [ ]:
from pathlib import Path
import copy
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
    precision_recall_curve,
)

SEED = 42
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15  # test gets the remaining 15%
BATCH_SIZE = 64
MAX_EPOCHS = 40
PATIENCE = 8
LEARNING_RATE = 1e-3
SUPERVISED_LOSS_WEIGHT = 1.0
DATA_PATH = Path("secom_telemetry_modified.csv")

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception:
    pass
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Looking for dataset: {DATA_PATH.resolve()}")

## 1. Load, identify the label, and establish time order

The notebook supports common SECOM label encodings (`-1/+1` and `0/1`) and explicit pass/fail strings. For the conventional SECOM encoding, `+1` is treated as defective and `-1` as non-defective. Ambiguous labels stop with an error instead of silently guessing.

In [ ]:
def canonical_column(name):
    name = str(name).strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return re.sub(r"_+", "_", name).strip("_")

TARGET_CANDIDATES = {
    "label", "target", "class", "y", "pass_fail", "passfail", "failure",
    "defect", "defective", "is_defective", "target_label", "wafer_status",
    "yield_status", "quality_label", "outcome",
}
TIME_CANDIDATES = {
    "timestamp", "datetime", "date", "date_time", "event_time", "measurement_time",
    "recorded_at", "created_at", "wafer_time", "sample_time", "time_index",
    "sequence", "sequence_id", "acquisition_time", "time",
}
ID_CANDIDATES = {
    "id", "wafer_id", "sample_id", "row_id", "record_id", "lot_id",
    "batch_id", "index", "row_index", "unnamed_0",
}


def find_target_column(columns):
    cols = list(columns)
    exact = [c for c in cols if c in TARGET_CANDIDATES]
    if len(exact) == 1:
        return exact[0]
    if len(exact) > 1:
        raise ValueError(f"Multiple possible target columns found: {exact}. Set the target column explicitly.")
    # A cautious fallback for names such as binary_defect_label.
    fuzzy = [c for c in cols if any(k in c for k in ("label", "defect", "pass_fail", "failure", "target"))]
    if len(fuzzy) == 1:
        return fuzzy[0]
    raise ValueError(f"Could not identify one target column. Available columns: {cols[:40]}")


def map_defect_labels(series):
    raw = series.copy()
    numeric = pd.to_numeric(raw, errors="coerce")
    nonmissing = raw.notna()
    if nonmissing.any() and numeric[nonmissing].notna().all():
        values = set(numeric[nonmissing].unique().tolist())
        if values.issubset({0, 1, 0.0, 1.0}):
            mapped = numeric
        elif values.issubset({-1, 1, -1.0, 1.0}):
            # Canonical UCI SECOM semantics: +1 = fail/defect, -1 = pass.
            mapped = (numeric == 1).astype(float)
        else:
            raise ValueError(
                f"Ambiguous numeric target values {sorted(values)[:20]}. "
                "Only 0/1 or conventional SECOM -1/+1 are inferred automatically."
            )
        return mapped.where(nonmissing, np.nan).astype("float")

    normalised = raw.astype("string").str.strip().str.lower().str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_")
    positives = {"1", "true", "yes", "y", "fail", "failed", "defect", "defective", "reject", "ng", "bad", "positive"}
    negatives = {"0", "false", "no", "n", "pass", "passed", "good", "ok", "accepted", "negative", "non_defect", "non_defective"}
    seen = set(normalised[normalised.notna()].unique().tolist())
    unknown = seen - positives - negatives
    if unknown:
        raise ValueError(f"Ambiguous string target values: {sorted(unknown)[:20]}")
    mapped = normalised.map(lambda v: 1.0 if v in positives else (0.0 if v in negatives else np.nan))
    return mapped.astype("float")


def find_time_column(frame, target_col):
    candidates = [c for c in frame.columns if c != target_col and c in TIME_CANDIDATES]
    if not candidates:
        return None, None
    # Prefer the most explicit timestamp-like column over a generic sequence/time field.
    priority = ["timestamp", "datetime", "date_time", "acquisition_time", "measurement_time", "event_time", "recorded_at", "created_at", "wafer_time", "sample_time", "time_index", "sequence", "sequence_id", "time"]
    candidates.sort(key=lambda c: priority.index(c) if c in priority else 999)
    for col in candidates:
        raw = frame[col]
        numeric_values = pd.to_numeric(raw, errors="coerce")
        if numeric_values.notna().all():
            return col, numeric_values
        parsed = pd.to_datetime(raw, errors="coerce", utc=True)
        if parsed.notna().all():
            return col, parsed
    return None, None


if not DATA_PATH.exists():
    candidates = [Path("/content/secom_telemetry_modified.csv"), Path("/mnt/data/secom_telemetry_modified.csv")]
    DATA_PATH = next((p for p in candidates if p.exists()), DATA_PATH)
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset not found. Place the supplied secom_telemetry_modified.csv beside this notebook "
        "or set DATA_PATH to its local path. Do not substitute a synthetic dataset for the final metrics."
    )

# Read values as strings so pandas does not infer feature types from the full dataset.
df = pd.read_csv(DATA_PATH, dtype="string")
df.columns = [canonical_column(c) for c in df.columns]
if len(set(df.columns)) != len(df.columns):
    raise ValueError("Column names collide after normalisation. Rename ambiguous columns before proceeding.")

target_col = find_target_column(df.columns)
y_all = map_defect_labels(df[target_col])
valid_label_rows = y_all.notna()
df = df.loc[valid_label_rows].copy().reset_index(drop=True)
y_all = y_all.loc[valid_label_rows].astype(int).reset_index(drop=True)

# Prefer a fully parseable timestamp or sequence field; otherwise preserve CSV row order.
time_col, order_values = find_time_column(df, target_col)
if time_col is not None and order_values is not None:
    df["__causal_order__"] = np.asarray(order_values)
    order_col = "__causal_order__"
    df = df.sort_values(order_col, kind="mergesort").reset_index(drop=True)
    # Keep labels aligned with the sorted rows.
    y_all = map_defect_labels(df[target_col]).astype(int).reset_index(drop=True)
    order_description = f"sorted by '{time_col}' (parsed successfully for every labelled row)"
else:
    order_col = None
    order_description = "CSV row order (no fully parseable time/sequence field found)"
    print("WARNING: no completely parseable time/sequence column found; confirm CSV row order is acquisition order.")

# Drop rows without labels before the temporal split; no labels are imputed.
y = y_all.to_numpy(dtype=np.int64)
if len(y) < 100:
    warnings.warn(f"Only {len(y)} labelled rows are available; estimates may be unstable.")
if np.unique(y).size != 2:
    raise ValueError("Both non-defect (0) and defect (1) labels are required.")

n = len(df)
train_end = int(math.floor(n * TRAIN_FRACTION))
val_end = int(math.floor(n * (TRAIN_FRACTION + VALIDATION_FRACTION)))
if train_end < 2 or val_end <= train_end or val_end >= n:
    raise ValueError(f"Not enough rows for a chronological split: n={n}")
train_idx = np.arange(0, train_end)
val_idx = np.arange(train_end, val_end)
test_idx = np.arange(val_end, n)
assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(val_idx) & set(test_idx)) == 0
assert train_idx[-1] < val_idx[0] < val_idx[-1] < test_idx[0]
if order_col is not None:
    ordered_values = df[order_col].to_numpy()
    assert np.all(ordered_values[:-1] <= ordered_values[1:]), "Rows are not monotonically ordered before splitting."

def check_both_classes(name, labels):
    counts = np.bincount(np.asarray(labels, dtype=int), minlength=2)
    if np.any(counts == 0):
        raise ValueError(f"{name} window has class counts {counts.tolist()}; chronological metrics cannot be estimated. Do not shuffle the final test set to fix this.")
    return counts

train_counts = check_both_classes("Training", y[train_idx])
val_counts = check_both_classes("Validation", y[val_idx])
test_counts = check_both_classes("Test", y[test_idx])

# Infer numeric telemetry from training rows only. Metadata and labels are never features.
exclude = set(TARGET_CANDIDATES) | set(ID_CANDIDATES) | set(TIME_CANDIDATES)
exclude.add(target_col)
if order_col:
    exclude.add(order_col)
feature_cols = []
train_schema = df.iloc[train_idx]
for col in df.columns:
    if col in exclude or col.startswith("unnamed") or col.startswith("__"):
        continue
    if col in {"sensor_0", "sensor_1"}:
        feature_cols.append(col)
        continue
    train_values = pd.to_numeric(train_schema[col], errors="coerce").to_numpy(dtype=float, na_value=np.nan)
    if np.isfinite(train_values).any():
        feature_cols.append(col)
if not feature_cols:
    raise ValueError("No numeric telemetry features found after excluding target, time, and identifiers.")

X_raw = df[feature_cols].copy()
X_train_raw = X_raw.iloc[train_idx].reset_index(drop=True)
X_val_raw = X_raw.iloc[val_idx].reset_index(drop=True)
X_test_raw = X_raw.iloc[test_idx].reset_index(drop=True)
y_train = y[train_idx]
y_val = y[val_idx]
y_test = y[test_idx]

print(f"Dataset rows after dropping missing labels: {n:,}")
print(f"Label column: {target_col!r}; order: {order_description}")
print(f"Raw feature columns: {len(feature_cols)}")
print(f"Split sizes: train={len(train_idx):,}, validation={len(val_idx):,}, test={len(test_idx):,}")
print(f"Overall defect prevalence: {y.mean():.3%}")
if abs(float(y.mean()) - 0.066) > 0.025:
    print("WARNING: overall defect prevalence differs materially from the expected ~6.6%; verify label mapping and the exact assessment CSV.")
print(f"Train/validation/test defect prevalence: {y_train.mean():.3%} / {y_val.mean():.3%} / {y_test.mean():.3%}")
print(f"Class counts (non-defect, defect): train={train_counts.tolist()}, val={val_counts.tolist()}, test={test_counts.tolist()}")

## 2. Causality-safe preprocessing and PS-L invariant handling

All medians, means, and standard deviations below are learned from the training window only. `sensor_0 == 0.0` is never used as a physical reading. On PS-L rows, `sensor_1` is interpreted only as a firmware category (`100`, `200`, or `300`); those codes never enter a continuous scaler. The continuous channels are neutral-zero placeholders when not applicable, while explicit mask/category features preserve the meaning.

In [ ]:
class CausalTelemetryPreprocessor:
    """Train-window-only imputation/scaling with explicit PS-L masks and firmware categories."""
    FIRMWARE_CODES = (100.0, 200.0, 300.0)

    def __init__(self, feature_columns):
        self.feature_columns = list(feature_columns)
        self.generic_cols_ = []
        self.generic_stats_ = {}
        self.special_stats_ = {}
        self.feature_names_out_ = []
        self.fitted_ = False

    @staticmethod
    def _numeric(frame, col):
        if col not in frame.columns:
            return pd.Series(np.nan, index=frame.index, dtype=float)
        return pd.to_numeric(frame[col], errors="coerce").astype(float)

    @staticmethod
    def _fit_stats(series):
        values = np.asarray(series, dtype=float)
        finite = np.isfinite(values)
        vals = values[finite]
        if vals.size == 0:
            return None
        median = float(np.median(vals))
        filled = np.where(finite, values, median)
        mean = float(np.mean(filled))
        scale = float(np.std(filled, ddof=0))
        if not np.isfinite(scale) or scale < 1e-8:
            scale = 1.0
        return {"median": median, "mean": mean, "scale": scale, "n_fit": int(vals.size)}

    def fit(self, frame):
        train = frame[self.feature_columns].copy()
        s0 = self._numeric(train, "sensor_0") if "sensor_0" in self.feature_columns else pd.Series(np.nan, index=train.index)
        legacy = s0.eq(0.0).fillna(False).to_numpy()

        # Fit each special channel only where its continuous measurement is valid.
        if "sensor_0" in self.feature_columns:
            vals0 = s0.mask(s0.eq(0.0))
            st0 = self._fit_stats(vals0)
            self.special_stats_["sensor_0"] = st0
        if "sensor_1" in self.feature_columns:
            s1 = self._numeric(train, "sensor_1")
            vals1 = s1.mask(legacy).mask(s1.isin(self.FIRMWARE_CODES) & legacy)
            self.special_stats_["sensor_1"] = self._fit_stats(vals1)

        reserved = {"sensor_0", "sensor_1"}
        for col in self.feature_columns:
            if col in reserved:
                continue
            st = self._fit_stats(self._numeric(train, col))
            if st is not None:
                self.generic_cols_.append(col)
                self.generic_stats_[col] = st

        names = []
        if "sensor_0" in self.feature_columns and self.special_stats_.get("sensor_0") is not None:
            names += ["sensor_0_scaled", "sensor_0_missing_nonlegacy"]
        if "sensor_1" in self.feature_columns:
            if self.special_stats_.get("sensor_1") is not None:
                names += ["sensor_1_continuous_scaled", "sensor_1_missing_nonlegacy"]
            names += ["ps_l_firmware_100", "ps_l_firmware_200", "ps_l_firmware_300", "ps_l_firmware_unknown_or_missing"]
        if "sensor_0" in self.feature_columns:
            names += ["ps_l_legacy_probe_station"]
        for col in self.generic_cols_:
            names += [f"{col}_scaled", f"{col}_missing"]
        self.feature_names_out_ = names
        self.fitted_ = True
        return self

    @staticmethod
    def _transform_continuous(series, stats, force_neutral_mask=None):
        values = np.asarray(series, dtype=float)
        missing = ~np.isfinite(values)
        filled = np.where(missing, stats["median"], values)
        scaled = (filled - stats["mean"]) / stats["scale"]
        # Bound extreme standardized values without fitting anything on held-out rows.
        scaled = np.clip(scaled, -8.0, 8.0)
        if force_neutral_mask is not None:
            scaled[np.asarray(force_neutral_mask, dtype=bool)] = 0.0
        return scaled.astype(np.float32), missing.astype(np.float32)

    def transform(self, frame):
        if not self.fitted_:
            raise RuntimeError("Call fit on the training window before transform.")
        data = frame[self.feature_columns].copy()
        parts = {}
        legacy = np.zeros(len(data), dtype=bool)
        if "sensor_0" in self.feature_columns:
            s0 = self._numeric(data, "sensor_0")
            legacy = s0.eq(0.0).fillna(False).to_numpy()
            stats0 = self.special_stats_.get("sensor_0")
            if stats0 is not None:
                s0_cont = s0.mask(s0.eq(0.0))
                vals, missing = self._transform_continuous(s0_cont, stats0, force_neutral_mask=legacy)
                # The continuous channel is inapplicable on legacy rows.
                missing[legacy] = 0.0
                parts["sensor_0_scaled"] = vals
                parts["sensor_0_missing_nonlegacy"] = missing

        if "sensor_1" in self.feature_columns:
            s1 = self._numeric(data, "sensor_1")
            is_legacy_code = legacy
            stats1 = self.special_stats_.get("sensor_1")
            if stats1 is not None:
                # Firmware codes are excluded from continuous scaling.
                s1_cont = s1.mask(is_legacy_code)
                vals, missing = self._transform_continuous(s1_cont, stats1, force_neutral_mask=is_legacy_code)
                missing[is_legacy_code] = 0.0
                parts["sensor_1_continuous_scaled"] = vals
                parts["sensor_1_missing_nonlegacy"] = missing
            known_code = np.zeros(len(data), dtype=bool)
            for code in self.FIRMWARE_CODES:
                mask = is_legacy_code & s1.eq(code).fillna(False).to_numpy()
                parts[f"ps_l_firmware_{int(code)}"] = mask.astype(np.float32)
                known_code |= mask
            unknown_code = is_legacy_code & ~known_code
            parts["ps_l_firmware_unknown_or_missing"] = unknown_code.astype(np.float32)

        if "sensor_0" in self.feature_columns:
            parts["ps_l_legacy_probe_station"] = legacy.astype(np.float32)

        for col in self.generic_cols_:
            raw = self._numeric(data, col)
            vals, missing = self._transform_continuous(raw, self.generic_stats_[col])
            parts[f"{col}_scaled"] = vals
            parts[f"{col}_missing"] = missing

        output = pd.DataFrame(parts, index=data.index)
        output = output.reindex(columns=self.feature_names_out_)
        arr = output.to_numpy(dtype=np.float32)
        if not np.isfinite(arr).all():
            raise ValueError("Preprocessing created non-finite values; inspect training feature columns and missingness.")
        return arr

    def fit_transform(self, frame):
        return self.fit(frame).transform(frame)


preprocessor = CausalTelemetryPreprocessor(feature_cols)
X_train = preprocessor.fit_transform(X_train_raw)
X_val = preprocessor.transform(X_val_raw)
X_test = preprocessor.transform(X_test_raw)

assert X_train.ndim == X_val.ndim == X_test.ndim == 2
assert X_train.shape[1] == X_val.shape[1] == X_test.shape[1] == len(preprocessor.feature_names_out_)
assert np.isfinite(X_train).all() and np.isfinite(X_val).all() and np.isfinite(X_test).all()

# Keep the firmware categories separate from continuous features.
if "sensor_1" in feature_cols:
    fw_names = [f"ps_l_firmware_{c}" for c in (100, 200, 300)]
    assert all(name in preprocessor.feature_names_out_ for name in fw_names)
    assert not any("firmware" in name and name.endswith("scaled") for name in preprocessor.feature_names_out_)
if "sensor_0" in feature_cols:
    assert "ps_l_legacy_probe_station" in preprocessor.feature_names_out_

print(f"Prepared matrices: train={X_train.shape}, validation={X_val.shape}, test={X_test.shape}")
print(f"Preprocessor fitted on {len(X_train):,} training rows only.")
if "sensor_0" in feature_cols:
    print(f"PS-L sentinel rows by split: train={int((X_train_raw['sensor_0'].eq(0.0)).sum())}, "
          f"validation={int((X_val_raw['sensor_0'].eq(0.0)).sum())}, test={int((X_test_raw['sensor_0'].eq(0.0)).sum())}")

## 3. Model and shape-safe objective

The model has a reconstruction head and a supervised defect-classification head. Both losses are reduced to exactly one scalar per wafer before combining. This prevents the silent `[batch, 1]` versus `[batch]` broadcast that can create a `[batch, batch]` loss matrix.

In [ ]:
class SupervisedAutoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        hidden = int(min(256, max(64, 2 ** int(np.ceil(np.log2(max(2, input_dim)))) // 2)))
        latent = int(min(48, max(8, input_dim // 8)))
        classifier_hidden = max(8, latent // 2)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.LayerNorm(hidden), nn.ReLU(), nn.Dropout(0.10),
            nn.Linear(hidden, latent), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent, hidden), nn.ReLU(), nn.Linear(hidden, input_dim),
        )
        self.classifier = nn.Sequential(
            nn.Linear(latent, classifier_hidden), nn.ReLU(), nn.Dropout(0.10), nn.Linear(classifier_hidden, 1),
        )

    def forward(self, x):
        z = self.encoder(x)
        reconstructed = self.decoder(z)
        logits = self.classifier(z).reshape(-1)
        return reconstructed, logits


def tensor_dataset(features, labels):
    x_tensor = torch.as_tensor(features, dtype=torch.float32)
    y_tensor = torch.as_tensor(np.asarray(labels).reshape(-1), dtype=torch.float32)
    assert x_tensor.ndim == 2
    assert y_tensor.ndim == 1
    assert x_tensor.shape[0] == y_tensor.shape[0]
    return TensorDataset(x_tensor, y_tensor)

train_dataset = tensor_dataset(X_train, y_train)
val_dataset = tensor_dataset(X_val, y_val)
test_dataset = tensor_dataset(X_test, y_test)

# Balance the training batches only; validation and test remain untouched.
train_counts = np.bincount(y_train.astype(int), minlength=2)
if np.any(train_counts == 0):
    raise ValueError(f"Cannot balance training batches; class counts are {train_counts.tolist()}.")
per_class_weight = 1.0 / train_counts
sample_weights = per_class_weight[y_train.astype(int)]
sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights), replacement=True,
    generator=torch.Generator().manual_seed(SEED),
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, drop_last=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

model = SupervisedAutoencoder(input_dim=X_train.shape[1]).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
bce_per_row = nn.BCEWithLogitsLoss(reduction="none")


def calculate_batch_loss(reconstruction, logits, inputs, labels):
    # Compute one reconstruction loss and one classification loss per wafer.
    assert reconstruction.shape == inputs.shape, (reconstruction.shape, inputs.shape)
    logits = logits.reshape(-1)
    labels = labels.reshape(-1)
    assert logits.ndim == labels.ndim == 1
    assert logits.shape[0] == labels.shape[0] == inputs.shape[0]
    recon_per_row = (reconstruction - inputs).pow(2).mean(dim=1)  # [batch]
    bce_per_row_values = bce_per_row(logits, labels)               # [batch]
    assert recon_per_row.shape == bce_per_row_values.shape == labels.shape
    total_per_row = recon_per_row + SUPERVISED_LOSS_WEIGHT * bce_per_row_values
    return total_per_row.mean(), recon_per_row.mean().detach(), bce_per_row_values.mean().detach()


def evaluate_loss(net, loader):
    net.eval()
    totals, recon_values, bce_values, n_seen = 0.0, 0.0, 0.0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            reconstructed, logits = net(xb)
            loss, rec, bce = calculate_batch_loss(reconstructed, logits, xb, yb)
            count = len(xb)
            totals += float(loss.item()) * count
            recon_values += float(rec.item()) * count
            bce_values += float(bce.item()) * count
            n_seen += count
    return {"loss": totals / n_seen, "reconstruction_loss": recon_values / n_seen, "bce_loss": bce_values / n_seen}


history = []
best_val_bce_loss = float("inf")
best_state = None
stale_epochs = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    train_total, train_recon, train_bce, n_seen = 0.0, 0.0, 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        reconstructed, logits = model(xb)
        loss, rec, bce = calculate_batch_loss(reconstructed, logits, xb, yb)
        if not torch.isfinite(loss):
            raise FloatingPointError(f"Non-finite training loss at epoch {epoch}.")
        loss.backward()
        # Check gradients before updating weights.
        grad_tensors = [p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
        if not grad_tensors or not all(torch.isfinite(g).all().item() for g in grad_tensors):
            raise FloatingPointError(f"Missing or non-finite gradients at epoch {epoch}.")
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        count = len(xb)
        train_total += float(loss.item()) * count
        train_recon += float(rec.item()) * count
        train_bce += float(bce.item()) * count
        n_seen += count

    val_metrics = evaluate_loss(model, val_loader)
    row = {
        "epoch": epoch,
        "train_loss": train_total / n_seen,
        "train_reconstruction_loss": train_recon / n_seen,
        "train_bce_loss": train_bce / n_seen,
        "val_loss": val_metrics["loss"],
        "val_reconstruction_loss": val_metrics["reconstruction_loss"],
        "val_bce_loss": val_metrics["bce_loss"],
    }
    history.append(row)
    print(f"Epoch {epoch:02d} | train={row['train_loss']:.4f} | val={row['val_loss']:.4f} "
          f"(recon={val_metrics['reconstruction_loss']:.4f}, BCE={val_metrics['bce_loss']:.4f})")

    if val_metrics["bce_loss"] < best_val_bce_loss - 1e-5:
        best_val_bce_loss = val_metrics["bce_loss"]
        best_state = copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})
        stale_epochs = 0
    else:
        stale_epochs += 1
        if stale_epochs >= PATIENCE:
            print(f"Early stopping after {epoch} epochs; best validation BCE={best_val_bce_loss:.4f}.")
            break

if best_state is None:
    raise RuntimeError("Training did not produce a valid checkpoint.")
model.load_state_dict(best_state)
model.to(DEVICE)
history_df = pd.DataFrame(history)
display(history_df.tail(10))

## 4. Threshold selection and natural-prevalence test evaluation

The validation and test windows are not resampled. A decision threshold is selected using the **chronological validation window** only, then frozen. Final metrics are computed once on every row in the untouched chronological test window at its natural defect prevalence.

In [ ]:
def predict_probabilities(net, features):
    net.eval()
    chunks = []
    with torch.no_grad():
        for start in range(0, len(features), BATCH_SIZE):
            xb = torch.as_tensor(features[start:start + BATCH_SIZE], dtype=torch.float32, device=DEVICE)
            _, logits = net(xb)
            logits = logits.reshape(-1)
            assert logits.shape[0] == xb.shape[0]
            chunks.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(chunks).reshape(-1)

val_probabilities = predict_probabilities(model, X_val)
# Choose the F1 threshold on validation data only.
val_precision, val_recall, val_thresholds = precision_recall_curve(y_val, val_probabilities)
if len(val_thresholds) == 0:
    raise ValueError("Validation probabilities did not yield candidate thresholds.")
val_f1_candidates = 2 * val_precision[:-1] * val_recall[:-1] / (val_precision[:-1] + val_recall[:-1] + 1e-12)
best_threshold_idx = int(np.nanargmax(val_f1_candidates))
DECISION_THRESHOLD = float(val_thresholds[best_threshold_idx])
validation_best_f1 = float(val_f1_candidates[best_threshold_idx])

# Score each test row once; do not resample the test window or tune on it.
test_probabilities = predict_probabilities(model, X_test)
test_predictions = (test_probabilities >= DECISION_THRESHOLD).astype(int)

metrics = {
    "dataset": DATA_PATH.name,
    "label_column": target_col,
    "ordering": order_description,
    "n_total_labelled": int(n),
    "n_train": int(len(y_train)),
    "n_validation": int(len(y_val)),
    "n_test": int(len(y_test)),
    "overall_defect_prevalence": float(np.mean(y)),
    "train_defect_prevalence": float(np.mean(y_train)),
    "validation_defect_prevalence": float(np.mean(y_val)),
    "test_defect_prevalence": float(np.mean(y_test)),
    "validation_selected_threshold": DECISION_THRESHOLD,
    "validation_best_f1_at_selected_threshold": validation_best_f1,
    "test_accuracy": float(accuracy_score(y_test, test_predictions)),
    "test_precision": float(precision_score(y_test, test_predictions, zero_division=0)),
    "test_recall": float(recall_score(y_test, test_predictions, zero_division=0)),
    "test_f1": float(f1_score(y_test, test_predictions, zero_division=0)),
    "test_balanced_accuracy": float(balanced_accuracy_score(y_test, test_predictions)),
    "test_roc_auc": float(roc_auc_score(y_test, test_probabilities)) if np.unique(y_test).size == 2 else None,
    "test_average_precision": float(average_precision_score(y_test, test_probabilities)) if np.unique(y_test).size == 2 else None,
    "test_confusion_matrix_labels_0_then_1": confusion_matrix(y_test, test_predictions, labels=[0, 1]).tolist(),
    "test_natural_prevalence_untouched": True,
    "test_rows_resampled": False,
    "preprocessing_fit_rows": int(len(y_train)),
}

print("\nFINAL TEST METRICS — untouched chronological test window")
print(f"Natural test defect prevalence: {metrics['test_defect_prevalence']:.3%} "
      f"({int(y_test.sum())}/{len(y_test)} defective wafers)")
print(f"Threshold selected on validation only: {DECISION_THRESHOLD:.6f}")
for key in ["test_accuracy", "test_precision", "test_recall", "test_f1", "test_balanced_accuracy", "test_roc_auc", "test_average_precision"]:
    value = metrics[key]
    print(f"{key.replace('test_', '').replace('_', ' ').title():24s}: {'N/A' if value is None else f'{value:.4f}'}")
print("Confusion matrix [ [TN, FP], [FN, TP] ]:")
print(np.asarray(metrics["test_confusion_matrix_labels_0_then_1"]))
print("\nTrain, validation, and test label prevalence:",
      f"{metrics['train_defect_prevalence']:.3%}, {metrics['validation_defect_prevalence']:.3%}, {metrics['test_defect_prevalence']:.3%}")

# Save the measured metrics and loss history.
metrics_path = Path("metrics.json")
metrics_path.write_text(json.dumps(metrics, indent=2, allow_nan=False) + "\n", encoding="utf-8")
history_df.to_csv("training_history.csv", index=False)
print(f"\nWrote {metrics_path.resolve()} and training_history.csv")

## 5. Automatically update the audit's measured-metrics section

This last cell inserts the actual measured values from this run into `AUDIT.md`. If the notebook is executed in a temporary environment that does not contain `AUDIT.md`, `metrics.json` remains the source of truth and the metric block can be copied into the audit.

In [ ]:
def update_audit_metrics(metrics_obj, audit_path=Path("AUDIT.md")):
    start_marker = "<!-- MEASURED_METRICS_START -->"
    end_marker = "<!-- MEASURED_METRICS_END -->"
    table_rows = [
        ("Natural test defect prevalence", f"{metrics_obj['test_defect_prevalence']:.3%}"),
        ("Test sample count", f"{metrics_obj['n_test']:,}"),
        ("Decision threshold (selected on validation)", f"{metrics_obj['validation_selected_threshold']:.6g}"),
        ("Precision", f"{metrics_obj['test_precision']:.4f}"),
        ("Recall", f"{metrics_obj['test_recall']:.4f}"),
        ("F1 score", f"{metrics_obj['test_f1']:.4f}"),
        ("Accuracy", f"{metrics_obj['test_accuracy']:.4f}"),
        ("Balanced accuracy", f"{metrics_obj['test_balanced_accuracy']:.4f}"),
        ("ROC-AUC", "N/A" if metrics_obj["test_roc_auc"] is None else f"{metrics_obj['test_roc_auc']:.4f}"),
        ("Average precision (PR-AUC summary)", "N/A" if metrics_obj["test_average_precision"] is None else f"{metrics_obj['test_average_precision']:.4f}"),
        ("Confusion matrix [ [TN, FP], [FN, TP] ]", f"`{metrics_obj['test_confusion_matrix_labels_0_then_1']}`"),
    ]
    block = start_marker + "\n" + "| Metric | Measured result |\n|---|---:|\n" + "\n".join(f"| {k} | {v} |" for k, v in table_rows) + "\n" + end_marker
    if not audit_path.exists():
        print("AUDIT.md not found in current working directory; actual metrics are saved in metrics.json.")
        return
    existing = audit_path.read_text(encoding="utf-8")
    if start_marker in existing and end_marker in existing:
        before = existing.split(start_marker, 1)[0]
        after = existing.split(end_marker, 1)[1]
        updated = before + block + after
    else:
        updated = existing.rstrip() + "\n\n## Measured evaluation metrics\n\n" + block + "\n"
    audit_path.write_text(updated, encoding="utf-8")
    print(f"Updated measured-metrics section in {audit_path.resolve()}")

update_audit_metrics(metrics)

# Confirm that the final metrics use the original test rows.
assert len(test_predictions) == len(y_test)
assert metrics["test_natural_prevalence_untouched"] is True
assert metrics["test_rows_resampled"] is False
print("Release checks passed: test rows remained untouched; reported metrics come from the model run above.")